# 🦋 나비효과 — Risk 분류기 v2 (GPU cuDF)
## 11.8M rows → 9개 risk_type 재분류

**입력**: `risk_events_final_v2.parquet` (11,834,729 rows, exploded 포맷)
**출력**: `risk_events_classified_v2.parquet`

**런타임**: GPU (T4) 필수 → `런타임 > 런타임 유형 변경 > T4 GPU`

**v1 대비 최적화**:
- URL dedup: 동일 기사 중복 분류 제거 (11.8M → 10.9M unique, ~8% 절약)
- 청크 저장: 메모리 효율적 parquet 저장
- Exploded 스키마 호환 (company_id per row)

**방법**: 119개 unified term (EN+CN+KR) × 9 카테고리, `cudf.Series.str.contains()` GPU 벡터 연산

## 1. 환경 설정 & cuDF 설치

In [ ]:
!nvidia-smi
import subprocess, sys

# cuDF 설치 (RAPIDS) — Colab T4 환경 기준
try:
    import cudf
    print(f"cuDF already installed: {cudf.__version__}")
except ImportError:
    print("Installing cuDF (RAPIDS)... ~2-3분 소요")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--extra-index-url=https://pypi.nvidia.com",
        "cudf-cu12", "pylibcudf-cu12",
        "-q"
    ])
    import cudf
    print(f"cuDF installed: {cudf.__version__}")

import cudf
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import time, gc
print("All imports OK")

## 2. Google Drive 마운트 & 데이터 로드

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

# ── 경로 설정 ──
DRIVE_BASE = "/content/drive/MyDrive/nabi"   # ← Drive 내 프로젝트 폴더
INPUT_PATH = f"{DRIVE_BASE}/risk_events_final_v2.parquet"
OUTPUT_PATH = f"{DRIVE_BASE}/risk_events_classified_v2.parquet"

import os
if not os.path.exists(INPUT_PATH):
    print(f"{INPUT_PATH} not found on Drive.")
    print("Drive에 risk_events_final_v2.parquet 업로드 후 이 셀 재실행")
    print("또는 아래 경로를 수정하세요.")
    raise FileNotFoundError(INPUT_PATH)

print(f"Found: {INPUT_PATH}")
fsize = os.path.getsize(INPUT_PATH) / 1024 / 1024
print(f"Size: {fsize:.0f} MB")

In [ ]:
# ── URL 기준 unique text 추출 (분류 대상) ──
# exploded 포맷이므로 같은 기사가 여러 행 존재 → dedup
t0 = time.time()
print("Step 1: URL dedup으로 unique 기사 추출...")

# 컬럼 최소한으로 로드 (메모리 절약)
pf = pq.ParquetFile(INPUT_PATH)
total_rows = pf.metadata.num_rows
print(f"Total rows: {total_rows:,}")

# url + risk_types만 로드해서 unique 추출
url_texts = {}  # url -> text_for_classification
for batch in pf.iter_batches(batch_size=500_000, columns=['url', 'risk_types']):
    df = batch.to_pandas()
    for url in df['url'].unique():
        if url not in url_texts:
            url_texts[url] = url  # title이 대부분 비어있으므로 url 사용
    del df

print(f"Unique URLs: {len(url_texts):,} (전체 {total_rows:,}의 {len(url_texts)/total_rows*100:.1f}%)")

# title이 있는 경우를 위해 재탐색
print("Step 2: title 있는 행 우선 수집...")
for batch in pf.iter_batches(batch_size=500_000, columns=['url', 'title']):
    df = batch.to_pandas()
    df['title'] = df['title'].fillna('').astype(str)
    has_title = df['title'].str.len() >= 5
    for _, row in df[has_title].iterrows():
        url_texts[row['url']] = row['title']  # title로 덮어쓰기
    del df

# Series로 변환
unique_urls = list(url_texts.keys())
unique_texts = pd.Series([url_texts[u] for u in unique_urls], dtype=str)
n_from_title = (unique_texts != pd.Series(unique_urls)).sum()
print(f"  title 사용: {n_from_title:,} | url fallback: {len(unique_urls)-n_from_title:,}")
print(f"  Loaded in {time.time()-t0:.1f}s")
del url_texts
gc.collect()

## 3. Risk Keywords 정의 (v5 — 119 unified terms, EN+CN+KR)

In [ ]:
# ── risk_keywords v5: 9 categories × 119 unified terms ──

RISK_CONFIG = {
    "geopolitics": {
        "terms": [
            ("airstrike",        5, r"airstrike|air strike|空袭|공습"),
            ("missile_strike",   5, r"missile strike|导弹.*袭击|미사일.*공격"),
            ("military_strike",  5, r"military strike|军事打击|군사.*타격"),
            ("bombing",          5, r"bombing|轰炸|폭격"),
            ("invasion",         5, r"invasion|入侵|침공"),
            ("armed_conflict",   5, r"armed conflict|武装冲突|军事冲突|무력충돌|군사충돌"),
            ("war",              5, r"war[^n]|trade.war|war.zone|civil.war|war.risk|战争|전쟁"),
            ("retaliation",      5, r"retaliation|retaliatory|报复|反制|보복조치|맞대응"),
            ("escalation",       5, r"escalation|escalate|局势升级|冲突升级|紧张局势|긴장고조|갈등심화|대립격화"),
            ("state_of_emergency",5, r"state of emergency|紧急状态|비상사태"),
            ("sanction",         4, r"sanction[s]?|制裁|经济制裁|제재|경제제재"),
            ("embargo",          4, r"embargo[es]?|禁运|금수조치"),
            ("export_control",   4, r"export control[s]?|export ban|export restriction|出口管制|出口限制|수출규제|수출통제|수입금지"),
            ("entity_list",      4, r"entity list|blacklist|blocked list|实体清单|黑名单|制裁名单|不可靠实体|블랙리스트|제재명단"),
            ("asset_freeze",     4, r"asset freeze|asset seizure|资产冻结|资产扣押|자산동결"),
            ("martial_law",      4, r"martial law|戒严|계엄"),
            ("resource_weapon",  4, r"resource weaponi[sz]ation|mineral weapon|critical mineral ban|资源武器化|稀土管制|자원무기화|핵심광물.*제한"),
            ("mineral_ban",      4, r"lithium.*ban|cobalt.*ban|graphite.*restriction|graphite.*ban|锂.*禁|钴.*禁|石墨.*管制|리튬.*규제|희토류.*규제|광물.*수출.*금지"),
            ("tariff",           3, r"tariff[s]?|关税|加征关税|反倾销|반덤핑|관세|보복관세|상계관세"),
            ("trade_war",        3, r"trade war|贸易战|贸易摩擦|贸易冲突|무역전쟁|무역분쟁|통상분쟁"),
            ("diplomatic_crisis",3, r"diplomatic crisis|diplomatic tension|外交危机|외교.*위기|외교.*갈등"),
            ("ceasefire",        3, r"ceasefire|cease-fire|停火|휴전"),
            ("national_security",3, r"national security|国家安全|国防安全|안보|국가안보"),
            ("terror_attack",    3, r"terror attack|terrorist attack|恐怖袭击|테러.*공격"),
            ("geopolitical_risk",3, r"geopolitical.*risk|地缘政治|중미갈등|미중갈등|지정학.*리스크|지정학적"),
        ]
    },
    "logistics": {
        "terms": [
            ("blockade",         5, r"blockade|封锁|海上封锁|봉쇄|해상봉쇄"),
            ("strait_closure",   5, r"strait closed|strait closure|strait blocked|航道封锁|항로차단|해협.*봉쇄"),
            ("shipping_halt",    5, r"shipping halted|shipping suspended|shipping stopped|停航|停运|航运.*中断|선박.*중단"),
            ("vessel_attack",    5, r"tanker attack|tanker struck|tanker seized|vessel attack|vessel seized|ship attacked|ship detained|drone attack.*ship|drone strike.*vessel|船舶.*袭击|선박.*공격"),
            ("hormuz",           4, r"Strait of Hormuz|Hormuz|霍尔木兹海峡|호르무즈해협"),
            ("suez",             4, r"Suez Canal|Suez|苏伊士运河|수에즈운하"),
            ("red_sea",          4, r"Red Sea|Bab el-Mandeb|红海|홍해"),
            ("malacca",          4, r"Malacca Strait|Strait of Malacca|马六甲海峡|말라카해협"),
            ("taiwan_strait",    4, r"Taiwan Strait|台湾海峡|대만해협"),
            ("port_disruption",  4, r"port strike|dock strike|longshoremen strike|port congestion|port backlog|port closure|港口罢工|港口拥堵|港口关闭|码头.*停工|항만파업|항만.*마비|부두.*파업"),
            ("freight_surge",    4, r"freight rate.*surge|freight rate.*spike|shipping cost.*surge|shipping cost.*rise|运费.*暴涨|运价.*飙升|海运.*涨价|운임.*급등|해운.*폭등"),
            ("insurance_surge",  4, r"insurance premium.*surge|war risk insurance|shipping insurance.*spike|保险.*暴涨|전쟁보험.*급등"),
            ("reroute",          4, r"reroute|re-route|diversion|alternative route|改道|绕行|우회항로"),
            ("supply_disruption",3, r"supply disruption|supply chain disruption|供应链中断|供应链断裂|供应中断|断供|缺货|공급망.*중단|공급망.*차질|공급.*부족|공급난|품귀"),
            ("shipment_delay",   3, r"delayed shipment[s]?|shipment delay|延误|船舶.*滞留|선적.*지연"),
            ("container_short",  3, r"container shortage|container crunch|集装箱.*短缺|컨테이너.*부족"),
            ("logistics_bottle", 3, r"logistics bottleneck|logistics disruption|物流.*瘫痪|물류.*대란|물류.*마비"),
            ("customs_delay",    3, r"customs delay|customs seizure|border closure|海关.*延误|통관.*지연"),
            ("transport_strike", 3, r"rail strike|railway strike|train strike|truck strike|trucker strike|铁路.*罢工|卡车.*罢工|철도.*파업|화물.*파업"),
            ("factory_shutdown", 3, r"factory shutdown|factory closure|plant shutdown|production halt|停产|停工|工厂.*关闭|生产.*中断|공장.*중단|공장.*폐쇄|생산.*중단|가동.*중단|조업.*중단"),
            ("power_outage",     3, r"power outage|blackout|power cut|停电|限电|拉闸限电|电力短缺|정전|전력부족|전력난|블랙아웃"),
        ]
    },
    "natural": {
        "terms": [
            ("earthquake",       5, r"earthquake|seismic|地震|震灾|余震|지진|진도|여진"),
            ("tsunami",          5, r"tsunami|tidal wave|海啸|쓰나미|해일"),
            ("typhoon",          5, r"typhoon|台风|强台风|태풍"),
            ("hurricane",        5, r"hurricane|飓风|허리케인"),
            ("flood",            4, r"flood|flooding|洪水|洪灾|暴雨.*灾|水灾|内涝|决堤|홍수|폭우|수해|침수|범람"),
            ("wildfire",         4, r"wildfire|forest fire|山火|森林火灾|林火|산불|산림화재"),
            ("landslide",        4, r"landslide|山体滑坡|泥石流|塌方|산사태|토사"),
            ("heatwave",         4, r"heatwave|heat wave|extreme heat|高温|热浪|极端高温|폭염|열파"),
            ("extreme_weather",  4, r"extreme weather|极端天气|恶劣天气|暴风雪|冰灾|寒潮|이상기후|극한기후|기상이변|한파|폭설"),
            ("drought",          3, r"drought|water shortage|water stress|干旱|旱灾|缺水|가뭄|물부족"),
            ("storm",            3, r"storm|暴风|폭풍"),
        ]
    },
    "regulatory": {
        "terms": [
            ("subsidy_cut",      4, r"subsidy cut|subsidy.*reduc|subsid.*phase.out|incentive.*cut|补贴.*退坡|补贴.*削减|补贴.*取消|보조금.*축소|보조금.*삭감|보조금.*폐지|인센티브.*축소"),
            ("antitrust",        4, r"antitrust|anti-trust|monopoly.*investigat|cartel|反垄断|垄断.*调查|价格.*操纵|공정위|반독점|독점.*조사|담합|불공정.*거래"),
            ("safety_recall",    4, r"recall|safety defect|NHTSA.*recall|fire risk|battery fire|召回|安全隐患|缺陷.*汽车|리콜|안전결함"),
            ("investigation",    4, r"investigat.*fraud|investigat.*bribery|SEC.*investigat|probe|立案|被查|违规|行政处罚|证监会.*处罚|수사|조사.*착수|검찰.*조사|감사.*적발|과징금"),
            ("FEOC",             4, r"FEOC|foreign entity of concern|entity of concern.*battery|外国敏感实体|우려외국법인"),
            ("nationalization",  4, r"nationali[sz]ation|nationali[sz]e.*lithium|nationali[sz]e.*mine|国有化|收归国有|矿产.*国有|국유화"),
            ("safety_accident",  4, r"safety accident|explosion|factory fire|安全事故|爆炸|火灾|工厂.*着火|车间.*起火|안전사고|폭발사고|화재사고|공장.*화재"),
            ("regulation",       3, r"regulation[s]?|regulatory|监管|法规|合规|政策.*收紧|규제|법규|정책.*강화|규제.*강화"),
            ("IRA",              3, r"Inflation Reduction Act|IRA.*battery|IRA.*EV|IRA.*clean energy|通胀削减法|IRA.*电池|인플레이션감축법|IRA.*배터리|IRA.*전기차"),
            ("emissions",        3, r"emission[s]? standard|carbon.*regulation|EU.*battery.*regulation|排放.*标准|碳.*规|배출.*기준"),
            ("EU_battery_reg",   3, r"EU Battery Regulation|battery passport|carbon footprint.*battery|due diligence.*battery|电池护照|배터리여권"),
            ("CBAM",             3, r"CBAM|Carbon Border Adjustment|carbon border tax|碳边境|탄소국경"),
            ("penalty_fine",     3, r"penalty|compliance.*violation|罚款|处罚|벌금|과태료"),
            ("env_violation",    3, r"environmental.*violation|pollution.*fine|toxic.*spill|waste.*dump|环保.*处罚|排放.*超标|环境.*违法|환경.*위반|배출.*초과|폐수|오염.*벌금"),
        ]
    },
    "market": {
        "terms": [
            ("bankruptcy",       5, r"bankrupt|insolvency|chapter 11|liquidation|wind.?down|破产|倒闭|清算|资不抵债|파산|부도|청산|법정관리|회생절차"),
            ("debt_crisis",      5, r"debt crisis|debt.*default|bond.*default|liquidity.*crisis|cash.*crunch|债务危机|流动性危机|资金链.*断裂|爆雷|暴雷|债务违约|유동성.*위기|자금난|부채.*위기|채무.*불이행|디폴트"),
            ("layoff",           4, r"layoff[s]?|lay off|mass firing|job cut[s]?|workforce reduction|裁员|大规模裁员|减员|缩编|구조조정|대량해고|감원|인력.*감축|희망퇴직"),
            ("demand_decline",   4, r"demand.*declin|demand.*drop|demand.*slump|sales.*plunge|sales.*drop|需求.*下降|需求.*萎缩|销量.*下滑|销量.*暴跌|订单.*减少|수요.*감소|수요.*위축|판매.*급감|수주.*감소|주문.*취소"),
            ("overcapacity",     4, r"overcapacity|excess capacity|oversupply|glut|产能过剩|供过于求|库存.*积压|공급과잉|과잉.*생산|재고.*누적|재고.*급증"),
            ("price_crash",      4, r"price.*crash|price.*plunge|price.*collapse|lithium.*price.*drop|cobalt.*price.*drop|nickel.*price.*drop|价格.*暴跌|价格.*崩盘|锂.*价格.*下跌|钴.*价格.*下跌|碳酸锂.*跌|가격.*폭락|가격.*급락|리튬.*가격.*하락|코발트.*가격.*하락|니켈.*가격.*하락"),
            ("credit_downgrade", 4, r"credit.*downgrade|rating.*downgrade|debt.*downgrade|default|信用.*降级|评级.*下调|신용.*하향|등급.*하향"),
            ("stock_crash",      4, r"stock.*crash|stock.*plunge|share.*crash|market.*crash|market.*selloff|sell-off|股价.*暴跌|股价.*崩盘|跌停|市值.*蒸发|주가.*폭락|주가.*급락|하한가|시가총액.*증발"),
            ("writedown",        4, r"write.?down|impairment|asset.*loss|goodwill.*loss|资产减值|商誉.*减值|자산.*손상|감액"),
            ("contract_cancel",  4, r"contract.*cancel|deal.*cancel|order.*cancel|contract.*terminat|deal.*collapse|订单.*取消|合同.*终止|合作.*中止|项目.*搁置|毁约|계약.*해지|계약.*취소|수주.*취소|프로젝트.*중단"),
            ("EV_slowdown",      4, r"EV.*slowdown|EV.*demand.*weak|electric vehicle.*demand.*fall|EV.*sales.*declin|新能源.*销量.*下降|电动车.*需求.*放缓|新能源.*增速.*放缓|전기차.*판매.*감소|전기차.*수요.*둔화|EV.*수요.*감소|전기차.*성장.*둔화"),
            ("delisting",        4, r"delist|delisting|removal from index|退市|摘牌|ST.*警告|상장폐지"),
            ("currency_crisis",  4, r"currency.*crash|currency.*crisis|devaluation|forex.*crisis|货币.*危机|贬值|환율.*급락|환율.*위기"),
            ("profit_warning",   3, r"profit warning|earnings.*miss|revenue.*miss|guidance.*cut|guidance.*lower|业绩.*预亏|利润.*下降|营收.*下滑|亏损|盈利.*下降|业绩.*暴雷|실적.*악화|적자|영업이익.*감소|매출.*감소|어닝쇼크"),
            ("restructuring",    3, r"restructuring|reorgani[sz]ation|cost.?cutting|downsizing|重组|债务重组|资产剥离|사업.*재편|자산.*매각|사업.*축소"),
            ("margin_squeeze",   3, r"margin.*squeeze|margin.*pressure|margin.*declin|profitability.*declin|毛利.*下降|利润率.*下滑|마진.*악화|수익성.*하락"),
            ("inventory_buildup",3, r"inventory.*build|inventory.*pile|inventory.*glut|channel.*stuff|库存.*积压|去库存|재고.*적체"),
        ]
    },
    "labor": {
        "terms": [
            ("mine_accident",    5, r"mine.*accident|mine.*collapse|mining.*disaster|mine.*explosion|矿难|矿山.*事故|矿.*坍塌|矿.*爆炸|透水事故|광산.*사고|광산.*붕괴|광산.*폭발"),
            ("worker_strike",    4, r"strike[s]?.*worker|worker.*strike|labor.*strike|union.*strike|罢工|工人.*抗议|员工.*维权|停工.*抗议|파업|노조.*파업|총파업|쟁의.*행위|노사분쟁"),
            ("workplace_safety", 4, r"workplace.*accident|industrial.*accident|worker.*death|occupational.*hazard|安全生产.*事故|工伤|工厂.*事故|车间.*事故|职业病|산업재해|산재|중대재해|작업장.*사고"),
            ("labor_shortage",   3, r"labor shortage|worker shortage|skills shortage|talent shortage|用工荒|招工难|人才.*短缺|劳动力.*不足|인력부족|구인난|인력난|노동력.*부족"),
            ("worker_protest",   3, r"protest.*factory|protest.*mine|protest.*plant|worker.*protest|工人.*抗议|工人.*示威|노동자.*시위|근로자.*항의"),
            ("union_dispute",    3, r"union.*dispute|union.*negotiat|collective.*bargain|labor.*dispute|劳资纠纷|工会.*谈判|노사.*갈등|단체교섭|노동.*분쟁"),
        ]
    },
    "technology": {
        "terms": [
            ("cyber_attack",     5, r"cyber.?attack|ransomware|data breach|hack|cyber.?security.*incident|malware|网络攻击|勒索软件|数据泄露|黑客|사이버.*공격|랜섬웨어|데이터.*유출|해킹|보안.*사고"),
            ("battery_defect",   5, r"battery.*defect|cell.*defect|thermal runaway|dendrite|short.?circuit.*battery|battery.*degradation|电池.*缺陷|热失控|电芯.*短路|电池.*衰减|배터리.*결함|열폭주|셀.*불량|배터리.*성능저하"),
            ("EV_fire",          5, r"EV.*fire|electric vehicle.*fire|electric car.*fire|vehicle.*fire.*battery|spontaneous.*combustion|电动车.*起火|新能源车.*自燃|电动汽车.*着火|电池.*自燃|전기차.*화재|전기차.*발화|전기차.*자연발화|EV.*화재"),
            ("patent_dispute",   4, r"patent.*dispute|patent.*infring|patent.*lawsuit|patent.*litigation|patent.*war|IP dispute|专利.*纠纷|专利.*侵权|专利.*诉讼|知识产权.*纠纷|특허.*분쟁|특허.*침해|특허.*소송|지식재산.*분쟁"),
            ("trade_secret",     4, r"trade secret|intellectual property.*theft|IP theft|technology.*theft|tech.*leak|技术.*泄露|기술.*유출"),
            ("technology_ban",   4, r"technology.*ban|tech.*restrict|chip.*ban|semiconductor.*restrict|技术封锁|技术.*禁令|芯片.*禁令|半导体.*限制|技术.*脱钩|기술.*제재|반도체.*규제"),
            ("tech_failure",     4, r"technolog.*fail|tech.*defect|software.*bug|system.*failure|quality.*defect|技术.*故障|系统.*故障|기술.*결함|시스템.*장애"),
            ("quality_issue",    3, r"quality.*issue|quality.*problem|product.*defect|manufacturing.*defect|QC.*fail|质量.*问题|质量.*缺陷|产品.*不合格|良率.*下降|품질.*문제|품질.*결함|불량률|제품.*하자"),
            ("RD_setback",       3, r"R&D.*fail|research.*setback|pilot.*fail|prototype.*fail|development.*delay|solid.?state.*delay|solid.?state.*setback|研发.*失败|研发.*受挫|연구.*실패|개발.*지연"),
        ]
    },
    "esg": {
        "terms": [
            ("human_rights",     5, r"human rights.*abuse|human rights.*violation|forced labo[u]r|modern slavery|Uyghur|Xinjiang.*labo[u]r|child labo[u]r.*cobalt|child labo[u]r.*mine|强迫劳动|童工|人权.*侵犯|新疆.*劳工|血汗工厂|인권.*침해|강제노동|아동노동|위구르.*노동"),
            ("governance_scandal",5,r"governance.*scandal|accounting.*fraud|embezzlement|insider.*trading|corporate.*fraud|bribery|corruption|财务造假|内幕交易|贪腐|行贿|侵吞|大股东.*占用|분식회계|내부자거래|횡령|배임|대주주.*사익편취|부정거래"),
            ("water_pollution",  4, r"water.*pollut|toxic.*discharge|chemical.*spill|tailings.*dam|acid.*mine.*drain|水污染|有毒.*排放|化学.*泄漏|尾矿.*泄漏|重金属.*超标|수질오염|유해물질.*유출|화학.*누출|중금속.*오염"),
            ("board_scandal",    4, r"CEO.*resign.*scandal|board.*oust|shareholder.*revolt|proxy.*fight|management.*turmoil|高管.*辞职|经营.*动荡|경영진.*사퇴|이사회.*갈등"),
            ("carbon_emission",  3, r"carbon.*emission|CO2.*emission|greenhouse.*gas|GHG.*emission|carbon footprint.*exceed|碳排放.*超标|碳足迹|温室气体|탄소배출.*초과|탄소발자국|온실가스"),
            ("greenwashing",     3, r"greenwash|green.?wash|false.*sustainab|misleading.*environmental|漂绿|그린워싱"),
            ("ESG_downgrade",    3, r"ESG.*downgrade|ESG.*rating.*cut|sustainab.*rating.*lower|ESG.*controversy|ESG.*评级.*下调|ESG.*争议|ESG.*등급.*하락|ESG.*논란"),
            ("deforestation",    3, r"deforestation|illegal.*logging|land.*grab|biodiversity.*loss|毁林|非法.*采伐|산림파괴|불법.*벌목"),
            ("supply_ethics",    3, r"supply chain.*ethic|supply chain.*transparency|traceability.*fail|conflict mineral|供应链.*透明|矿产.*冲突|공급망.*윤리|분쟁광물"),
        ]
    },
    "pandemic": {
        "terms": [
            ("pandemic",         5, r"pandemic|epidemic|outbreak|大流行|传染病|病毒.*爆发|팬데믹|전염병|감염병.*확산|대유행"),
            ("lockdown",         5, r"lockdown|lock-down|stay.?at.?home|shelter.?in.?place|movement.*restrict|封城|封控|静默管理|动态清零|全域静态管理|봉쇄|록다운|이동제한"),
            ("factory_lockdown", 5, r"factory.*lockdown|plant.*lockdown|production.*lockdown|zero.?covid|bubble.*manufactur|工厂.*封控|车间.*封闭|闭环生产|停产.*疫情|공장.*봉쇄|생산.*중단.*코로나|조업.*중단.*방역"),
            ("COVID",            4, r"COVID|coronavirus|SARS-CoV|covid.?19|新冠|新冠肺炎|冠状病毒|코로나|코로나19"),
            ("quarantine",       4, r"quarantine|isolation.*order|travel.*ban|border.*close.*virus|隔离|封闭管理|出行限制|集中隔离|격리|방역.*강화|입국.*제한|검역"),
            ("pandemic_supply",  4, r"pandemic.*supply|covid.*supply.*chain|lockdown.*supply|virus.*disrupt.*supply|疫情.*供应|复工复产|코로나.*공급망|방역.*물류"),
            ("variant",          3, r"variant.*concern|new.*variant|mutation.*virus|delta.*variant|omicron|变异.*毒株|변이.*바이러스"),
        ]
    },
}

total_terms = sum(len(v['terms']) for v in RISK_CONFIG.values())
print(f"Total: {total_terms} terms across {len(RISK_CONFIG)} categories")
for rtype, cfg in RISK_CONFIG.items():
    print(f"  {rtype:15s}: {len(cfg['terms']):3d} terms")

## 4. GPU 분류 실행 (cuDF) — Unique URL 기준

In [ ]:
t_total = time.time()
N = len(unique_texts)
print(f"Classifying {N:,} unique articles on GPU...")

# ── Step 1: cuDF로 전송 + lowercase ──
print("Transferring to GPU...")
t0 = time.time()
gdf_text = cudf.Series(unique_texts.values)
gdf_text = gdf_text.str.lower()
print(f"  GPU transfer + lowercase: {time.time()-t0:.1f}s")

# ── Step 2: 카테고리별 mega-regex 매칭 ──
print(f"\n{'='*50}")
print("GPU regex matching (9 categories)")
print(f"{'='*50}")
S_MAX = 25.0
RTYPES = list(RISK_CONFIG.keys())
match_results = {}
avg_weights = {}

for rtype in RTYPES:
    terms = RISK_CONFIG[rtype]['terms']
    mega = '|'.join(f'(?:{pat.lower()})' for _, _, pat in terms)
    avg_w = sum(w for _, w, _ in terms) / len(terms)
    avg_weights[rtype] = avg_w

    t0 = time.time()
    try:
        hit = gdf_text.str.contains(mega, regex=True)
    except Exception as e:
        print(f"  {rtype:15s}: cuDF regex error, pandas fallback...")
        hit_pd = unique_texts.str.lower().str.contains(mega, na=False, regex=True)
        hit = cudf.Series(hit_pd.values)

    n_hit = int(hit.sum())
    elapsed = time.time() - t0
    match_results[rtype] = hit
    print(f"  {rtype:15s}: {n_hit:>10,} hits ({n_hit/N*100:.2f}%) in {elapsed:.1f}s")

# ── Step 3: risk_types + severity 조합 ──
print("\nBuilding risk_types and severity...")
t0 = time.time()

match_pdf = pd.DataFrame({rt: match_results[rt].to_pandas() for rt in RTYPES})
del match_results, gdf_text
gc.collect()

# severity
severity_raw = pd.Series(0.0, index=range(N))
for rtype in RTYPES:
    severity_raw += match_pdf[rtype].astype(float) * avg_weights[rtype]
severity_arr = np.minimum(np.log(1 + severity_raw.values) / np.log(1 + S_MAX), 1.0).round(4)

# risk_types
any_match = match_pdf.any(axis=1)
risk_types_arr = np.full(N, 'other', dtype=object)
matched_idx = match_pdf.index[any_match]

def row_to_types(row):
    return ','.join(rt for rt in RTYPES if row[rt])

risk_types_arr[matched_idx] = match_pdf.loc[matched_idx].apply(row_to_types, axis=1).values

print(f"  Classified: {len(matched_idx):,} / {N:,} ({len(matched_idx)/N*100:.1f}%)")
print(f"  Combination done in {time.time()-t0:.1f}s")

# URL → 결과 매핑 딕셔너리
url_to_rtype = dict(zip(unique_urls, risk_types_arr))
url_to_severity = dict(zip(unique_urls, severity_arr))

elapsed_total = time.time() - t_total
print(f"\n{'='*50}")
print(f"GPU classification: {elapsed_total:.1f}s ({elapsed_total/60:.1f}min)")
print(f"{'='*50}")

del match_pdf, severity_raw, unique_texts
gc.collect()

## 5. 결과 매핑 & 저장 (청크 스트리밍)

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

# ── 결과 통계 (분류 전 집계) ──
from collections import Counter
rtype_counts = Counter(url_to_rtype.values())
print("="*60)
print("risk_types distribution (unique articles)")
print("="*60)
for rt, cnt in rtype_counts.most_common(25):
    bar = '#' * int(cnt / len(url_to_rtype) * 100)
    print(f"  {rt:40s}: {cnt:>10,} ({cnt/len(url_to_rtype)*100:5.1f}%) {bar}")

n_classified = sum(c for rt, c in rtype_counts.items() if rt != 'other')
n_other = rtype_counts.get('other', 0)
print(f"\n  Classified: {n_classified:,} ({n_classified/len(url_to_rtype)*100:.1f}%)")
print(f"  Other:      {n_other:,} ({n_other/len(url_to_rtype)*100:.1f}%)")

# 카테고리별 hit rate
print(f"\nPer-category hit rate:")
for rtype in RTYPES:
    n = sum(1 for v in url_to_rtype.values() if rtype in v)
    print(f"  {rtype:15s}: {n:>10,} ({n/len(url_to_rtype)*100:.2f}%)")

In [ ]:
# ── 청크 스트리밍으로 결과 매핑 & 저장 (메모리 절약) ──
print(f"\nMapping results to full dataset ({total_rows:,} rows)...")
print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

pf = pq.ParquetFile(INPUT_PATH)
writer = None
total_written = 0
t0 = time.time()

for batch in pf.iter_batches(batch_size=500_000):
    df = batch.to_pandas()
    del batch
    
    # URL로 분류 결과 매핑
    df['risk_types'] = df['url'].map(url_to_rtype).fillna('other')
    df['severity'] = df['url'].map(url_to_severity).fillna(0.0)
    
    # event_time tz-naive 보장
    if 'event_time' in df.columns:
        df['event_time'] = pd.to_datetime(df['event_time'], errors='coerce', utc=True)
        df['event_time'] = df['event_time'].dt.tz_localize(None)
    
    # finbert를 float로 통일
    df['finbert'] = pd.to_numeric(df.get('finbert'), errors='coerce')
    
    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
    writer.write_table(table)
    total_written += len(df)
    
    if total_written % 2_000_000 < 500_001:
        print(f"  {total_written:,} / {total_rows:,} ({total_written/total_rows*100:.0f}%)")
    
    del df, table
    gc.collect()

if writer:
    writer.close()

elapsed = time.time() - t0
print(f"\n✅ Saved: {OUTPUT_PATH}")
print(f"  {total_written:,} rows in {elapsed:.1f}s")
print(f"  Size: {os.path.getsize(OUTPUT_PATH)/1024/1024:.0f} MB")

## 6. 완료

**다음 단계:**
1. Drive에서 `risk_events_classified_v2.parquet` 다운로드
2. 로컬 프로젝트에서 `data/processed/risk_events.parquet`로 교체
3. Stage 6~10 재실행 (aggregation → KG → exposure → CAR → backtest)

```bash
# 로컬에서
cp risk_events_classified_v2.parquet data/processed/risk_events.parquet
python -m src.pipeline.run_remaining  # 또는 개별 스테이지 실행
```

In [ ]:
# ── (선택) CPU Fallback — cuDF 없을 때 ──
USE_CPU_FALLBACK = False  # ← True로 변경 후 실행

if USE_CPU_FALLBACK:
    print("Running CPU fallback (pandas str.contains)...")
    print(f"대상: {len(unique_urls):,} unique articles")
    print("예상 소요: ~30-60분")

    # unique_texts 재구성 (위에서 삭제된 경우)
    url_texts = {}
    pf = pq.ParquetFile(INPUT_PATH)
    for batch in pf.iter_batches(batch_size=500_000, columns=['url', 'title']):
        df = batch.to_pandas()
        df['title'] = df['title'].fillna('').astype(str)
        for _, row in df.drop_duplicates('url').iterrows():
            if row['url'] not in url_texts:
                url_texts[row['url']] = row['title'] if len(row['title']) >= 5 else row['url']
        del df

    unique_urls_cpu = list(url_texts.keys())
    text_series = pd.Series([url_texts[u] for u in unique_urls_cpu], dtype=str).str.lower()

    url_to_rtype = {}
    url_to_severity = {}
    match_results_cpu = {}

    for rtype in RTYPES:
        terms = RISK_CONFIG[rtype]['terms']
        mega = '|'.join(f'(?:{pat})' for _, _, pat in terms)
        t0 = time.time()
        hit = text_series.str.contains(mega, case=False, na=False, regex=True)
        match_results_cpu[rtype] = hit
        print(f"  {rtype:15s}: {hit.sum():>10,} hits in {time.time()-t0:.1f}s")

    match_pdf = pd.DataFrame(match_results_cpu)
    severity_raw = pd.Series(0.0, index=range(len(text_series)))
    for rtype in RTYPES:
        avg_w = sum(w for _, w, _ in RISK_CONFIG[rtype]['terms']) / len(RISK_CONFIG[rtype]['terms'])
        severity_raw += match_pdf[rtype].astype(float) * avg_w

    severity_arr = np.minimum(np.log(1 + severity_raw.values) / np.log(1 + S_MAX), 1.0).round(4)
    any_match = match_pdf.any(axis=1)
    risk_arr = np.full(len(text_series), 'other', dtype=object)
    matched = match_pdf.index[any_match]
    risk_arr[matched] = match_pdf.loc[matched].apply(row_to_types, axis=1).values

    url_to_rtype = dict(zip(unique_urls_cpu, risk_arr))
    url_to_severity = dict(zip(unique_urls_cpu, severity_arr))
    print(f"CPU fallback done. 이제 위의 '5. 결과 매핑 & 저장' 셀을 다시 실행하세요.")
else:
    print("CPU fallback disabled. cuDF 미지원 시 USE_CPU_FALLBACK = True로 변경")